# 03 - Pembuatan Submission Structural Damage Classification

**Mahasiswa:** Rahardian Putra  
**NRP:** 5052231018

Notebook tipis untuk membangun kembali berkas submission **tanpa melatih ulang**, memakai
`models/pipeline_lightgbm.joblib` yang dihasilkan oleh
`notebooks/00_workflow_lengkap_structural_damage.ipynb`.

> **Catatan namespace pickle.** Model disimpan dengan `joblib` dari notebook utama, sehingga
> kelas `BuildingFeatures` tercatat sebagai `__main__.BuildingFeatures`. Akibatnya model hanya
> dapat dimuat bila kelas tersebut sudah ada di sesi pemuatan. Karena itu kelas yang sama
> didefinisikan ulang di bawah ini **sebelum** `joblib.load` dipanggil. Ini juga alasan mengapa
> model tidak dapat dimuat langsung dari `scripts/` (di sana kelasnya bernama
> `scripts.feature_engineering.BuildingFeatures`).

Format `damage_grade` pada berkas submission adalah **bilangan bulat 1–5**, bukan string
`Grade N`. Skrip penilai kompetisi mem-*parse* kolom ini sebagai angka, sehingga mengirim
`Grade 1` menghasilkan *Evaluation Exception*.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class BuildingFeatures(BaseEstimator, TransformerMixin):
    """Feature engineering deterministik dari data pra-gempa, tanpa informasi target.

    Definisi ini harus identik dengan kelas pada notebook utama, karena `joblib`
    memuat ulang kelas berdasarkan nama ``__main__.BuildingFeatures``.
    """

    ID_COLUMNS = ("district_id", "vdcmun_id", "ward_id")

    def fit(self, X: pd.DataFrame, y=None):
        """Catat skema kolom masukan tanpa mempelajari nilai apa pun dari data."""
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Kembalikan fitur turunan dengan urutan baris yang dipertahankan."""
        available = [column for column in self.feature_names_in_ if column in X.columns]
        X = X.loc[:, available].copy()
        X = X.drop(columns=["building_id"], errors="ignore")
        for column in self.ID_COLUMNS:
            X[column] = X[column].astype("string").fillna("missing").astype(object)
        floors = X["count_floors_pre_eq"].replace(0, np.nan)
        X["height_per_floor"] = X["height_ft_pre_eq"] / floors
        X["estimated_floor_area"] = X["plinth_area_sq_ft"] * floors
        X["age_is_999"] = X["age_building"].eq(999).astype(int)
        X["log_plinth_area"] = np.log1p(X["plinth_area_sq_ft"].clip(lower=0))
        return X.replace([np.inf, -np.inf], np.nan)

In [ ]:
def find_data_dir() -> Path:
    """Cari folder yang memuat ``train.csv`` dan ``test.csv`` (Kaggle maupun lokal)."""
    candidates = [
        Path("/kaggle/input/structural-damage-classification"),
        Path("/kaggle/input/competitions/structural-damage-classification"),
        Path.cwd() / "data",
        Path.cwd(),
        Path.cwd().parent / "data",
    ]
    for folder in candidates:
        if (folder / "test.csv").exists() and (folder / "sample_submission.csv").exists():
            return folder
    for candidate in sorted(Path("/kaggle/input").glob("**/sample_submission.csv")):
        return candidate.parent
    raise FileNotFoundError("sample_submission.csv tidak ditemukan; atur DATA_DIR manual.")


DATA_DIR = find_data_dir()
ROOT = Path.cwd()
MODEL_PATH = ROOT / "models" / "pipeline_lightgbm.joblib"
LABELS = ["Grade 1", "Grade 2", "Grade 3", "Grade 4", "Grade 5"]
LABEL_TO_CODE = {label: index + 1 for index, label in enumerate(LABELS)}

assert MODEL_PATH.exists(), f"Model belum ada: {MODEL_PATH}. Jalankan notebook utama dulu."
model = joblib.load(MODEL_PATH)
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
print("Model  :", MODEL_PATH)
print("Test   :", test.shape, "| Sample:", sample.shape)

In [ ]:
def make_submission(model, test_frame: pd.DataFrame, sample_frame: pd.DataFrame,
                    output_path: Path, label_style: str = "numeric") -> pd.DataFrame:
    """Prediksi, selaraskan ke ID sample, validasi, lalu simpan berkas submission.

    Parameter
    ----------
    label_style : {"numeric", "grade"}
        ``"numeric"`` menulis ``damage_grade`` sebagai bilangan bulat 1-5 (dipakai
        untuk submission), ``"grade"`` menulis string ``"Grade 1"``-``"Grade 5"``.
    """
    if label_style not in ("numeric", "grade"):
        raise ValueError("label_style harus 'numeric' atau 'grade'.")
    if list(sample_frame.columns) != ["building_id", "damage_grade"]:
        raise ValueError("Kolom sample_submission tidak sesuai.")
    if len(sample_frame) != len(test_frame):
        raise ValueError("Jumlah baris test dan sample_submission berbeda.")
    if not test_frame.building_id.is_unique or not sample_frame.building_id.is_unique:
        raise ValueError("building_id harus unik.")
    if set(sample_frame.building_id) != set(test_frame.building_id):
        raise ValueError("Himpunan building_id test dan sample_submission tidak sama.")

    predicted_labels = pd.Series(model.predict(test_frame), index=test_frame.building_id)
    aligned = sample_frame.building_id.map(predicted_labels)

    if aligned.isna().any():
        raise ValueError("Ada bangunan yang belum terprediksi.")
    if not aligned.isin(LABELS).all():
        raise ValueError("Ada label di luar Grade 1-Grade 5.")

    submission = sample_frame.copy()
    if label_style == "numeric":
        submission["damage_grade"] = aligned.map(LABEL_TO_CODE).astype("int64")
    else:
        submission["damage_grade"] = aligned

    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(output_path, index=False)

    reread = pd.read_csv(output_path)
    pd.testing.assert_frame_equal(submission.reset_index(drop=True), reread)
    return submission


submission = make_submission(model, test, sample, ROOT / "submission.csv", label_style="numeric")
make_submission(model, test, sample, ROOT / "submission_grade_labels.csv", label_style="grade")

saved = pd.read_csv(ROOT / "submission.csv")
assert saved.damage_grade.dtype.kind in "iu", "damage_grade harus bilangan bulat."
assert set(saved.damage_grade.unique()) <= {1, 2, 3, 4, 5}, "Kode kelas harus 1-5."
print("Submission siap:", (ROOT / "submission.csv").resolve())
print("Bentuk:", saved.shape, "| Kolom:", list(saved.columns), "| Tipe:", saved.damage_grade.dtype)
display(saved.head())
display(saved.damage_grade.value_counts().sort_index().to_frame("jumlah prediksi"))